In [0]:
# Create Customers Data
customers_data = [
    (1, "Aravind", "BLR"),
    (2, "Raj", "HYD"),
    (3, "Priya", "CHN"),
    (4, "Anil", "BLR")
]

customers_df = spark.createDataFrame(customers_data, ["customer_id", "name", "city"])

# Save directly as table
customers_df.write.mode("overwrite").saveAsTable("bronze_customers")

In [0]:
%sql SELECT * FROM bronze_customers;


In [0]:
%sql
SELECT * FROM bronze_orders;

In [0]:

# Orders Data
orders_data = [
    (101, 1, 500),
    (102, 2, 300),
    (103, 1, 700),
    (104, 3, 200)
]

orders_df = spark.createDataFrame(orders_data, ["order_id", "customer_id", "amount"])

orders_df.write.mode("overwrite").saveAsTable("bronze_orders")


In [0]:
%sql
SELECT * FROM bronze_orders;

In [0]:
customers_df = spark.table("bronze_customers")
orders_df = spark.table("bronze_orders")

display(customers_df)
display(orders_df)

In [0]:

from pyspark.sql.functions import col

customers_clean = customers_df \
    .withColumn("customer_id", col("customer_id").cast("int"))

orders_clean = orders_df \
    .withColumn("order_id", col("order_id").cast("int")) \
    .withColumn("customer_id", col("customer_id").cast("int")) \
    .withColumn("amount", col("amount").cast("double"))


In [0]:

customers_clean = customers_clean.dropDuplicates(["customer_id"])
orders_clean = orders_clean.dropDuplicates(["order_id"])


In [0]:

silver_df = orders_clean.join(
    customers_clean,
    on="customer_id",
    how="inner"
)

display(silver_df)


In [0]:
silver_df.write.mode("overwrite").saveAsTable("silver_orders")

In [0]:
%sql
SELECT * FROM silver_orders;


In [0]:
silver_df = spark.table("silver_orders")
display(silver_df)

In [0]:
from pyspark.sql.functions import sum

total_sales = silver_df.groupBy("customer_id", "name") \
    .agg(sum("amount").alias("total_sales"))

display(total_sales)


In [0]:
top_customers = total_sales.orderBy("total_sales", ascending=False)

display(top_customers)


In [0]:
top_customers.write.mode("overwrite").saveAsTable("gold_top_customers")

In [0]:
%sql
SELECT * FROM gold_top_customers;